# Vulnerability Prioritisation

## What This Is
Organisations face thousands of CVEs per year but can only patch a fraction. ML-based vulnerability prioritisation ranks vulnerabilities by exploitation likelihood and impact, combining:

- **EPSS (Exploit Prediction Scoring System)**: ML model predicting 30-day exploitation probability, trained on actual exploitation data from CISA KEV and threat intelligence feeds
- **CVSS (Common Vulnerability Scoring System)**: Severity score based on impact, exploitability, scope
- **KEV (Known Exploited Vulnerabilities)**: CISA catalog of vulnerabilities actively exploited in the wild
- **Reachability**: Is the vulnerable component actually deployed and reachable in your environment?
- **Asset criticality**: Is the vulnerable host a critical production system or a dev workstation?

## EPSS vs CVSS
CVSS scores base rate for critical (9.0+): ~5% are ever exploited. EPSS targets actual exploitation probability — most critical CVEs are never exploited because they require specific conditions. EPSS has dramatically better prioritisation efficiency: top 2% by EPSS score covers ~88% of exploited vulnerabilities.

In [ ]:
import numpy as np
from typing import Dict, List, Tuple

np.random.seed(42)

# Simulate a vulnerability dataset with multi-signal features
N_VULNS = 500

def generate_vuln_dataset(n: int) -> Dict:
    """Simulate vulnerability features used for prioritisation."""
    # CVE characteristics
    cvss_base = np.random.choice([2, 3, 4, 5, 6, 7, 8, 9, 10], n, 
                                   p=[0.05, 0.05, 0.08, 0.10, 0.12, 0.15, 0.20, 0.15, 0.10])
    cvss_float = cvss_base + np.random.uniform(0, 0.9, n)
    
    # Exploit availability
    has_poc = np.random.binomial(1, 0.25, n)  # 25% have public PoC
    has_metasploit = has_poc * np.random.binomial(1, 0.3, n)  # 30% of PoC in MSF
    days_since_nvd = np.random.exponential(180, n).clip(1, 1000)
    
    # Threat intelligence
    in_kev = np.random.binomial(1, 0.08, n)   # 8% in CISA KEV
    epss_score = np.clip(
        0.001 + 0.3 * has_poc + 0.4 * in_kev + 0.1 * has_metasploit 
        + np.random.exponential(0.02, n), 0, 1
    )
    
    # Asset context
    asset_criticality = np.random.choice([1, 2, 3, 4, 5], n, p=[0.15, 0.25, 0.35, 0.15, 0.10])
    internet_facing = np.random.binomial(1, 0.3, n)
    in_pci_scope = np.random.binomial(1, 0.2, n)
    patch_available = np.random.binomial(1, 0.85, n)
    
    # Ground truth: was this actually exploited? (simulate)
    exploit_prob = np.clip(
        0.01 + 0.5 * in_kev + 0.2 * has_metasploit + 0.05 * internet_facing
        + 0.01 * (cvss_float / 10) + np.random.exponential(0.02, n), 0, 1
    )
    exploited = np.random.binomial(1, exploit_prob)
    
    return {
        'cvss': cvss_float, 'has_poc': has_poc, 'has_msf': has_metasploit,
        'days_old': days_since_nvd, 'in_kev': in_kev, 'epss': epss_score,
        'asset_crit': asset_criticality, 'internet': internet_facing,
        'pci': in_pci_scope, 'patch_avail': patch_available, 'exploited': exploited
    }

vulns = generate_vuln_dataset(N_VULNS)
print(f'Vulnerability dataset: {N_VULNS} CVEs')
print(f'  Actually exploited: {vulns["exploited"].sum()} ({vulns["exploited"].mean():.1%})')
print(f'  CVSS Critical (>=9): {(vulns["cvss"]>=9).sum()}')
print(f'  In CISA KEV: {vulns["in_kev"].sum()}')
print(f'  Have public PoC: {vulns["has_poc"].sum()}')


In [ ]:
# Compare prioritisation strategies

def prioritisation_efficiency(scores: np.ndarray, exploited: np.ndarray,
                               top_pct: float = 0.1) -> Dict:
    """What fraction of exploited vulns are in the top X%?"""
    n = len(scores)
    top_k = int(n * top_pct)
    top_idx = np.argsort(scores)[::-1][:top_k]
    
    total_exploited = exploited.sum()
    caught = exploited[top_idx].sum()
    
    return {
        'top_pct': top_pct,
        'queue_size': top_k,
        'exploited_caught': int(caught),
        'total_exploited': int(total_exploited),
        'detection_rate': caught / total_exploited if total_exploited > 0 else 0,
        'precision': caught / top_k if top_k > 0 else 0,
    }

# Multi-signal triage score
triage_score = (
    vulns['epss'] * 5 +
    vulns['cvss'] / 10 +
    vulns['in_kev'] * 3 +
    vulns['has_msf'] * 2 +
    vulns['internet'] * 1.5 +
    vulns['asset_crit'] / 5
)

strategies = [
    ('CVSS only', vulns['cvss']),
    ('EPSS only', vulns['epss']),
    ('KEV only', vulns['in_kev'].astype(float)),
    ('Multi-signal triage', triage_score),
]

print('Vulnerability Prioritisation Efficiency (Top 10%)')
print('=' * 65)
print(f'{'Strategy':<25} {'Queue Size':>11} {'Caught':>7} {'Det. Rate':>10} {'Precision':>10}')
print('-' * 65)

for name, scores in strategies:
    r = prioritisation_efficiency(scores, vulns['exploited'], top_pct=0.10)
    print(f'{name:<25} {r["queue_size"]:>11} {r["exploited_caught"]:>7} '
          f'{r["detection_rate"]:>10.3f} {r["precision"]:>10.3f}')

print('\nKey insight: Multi-signal triage detects more exploited vulns in a smaller queue')
print('than CVSS alone — because most critical CVSS scores are never actually exploited.')
